# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 4.4 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task165"
CH, H, W = 10, 30, 30
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [6]:
# Load task data. The bundle includes task165.json next to this notebook.
candidate_paths = [
    Path(COMPETITION)/f"{TASK_ID}.json",
]
task_path = None
for p in candidate_paths:
    if os.path.exists(p):
        task_path = p
        break
if task_path is None:
    raise FileNotFoundError(f"Could not find {TASK_ID}.json. Tried: {candidate_paths}")

with open(task_path, "r") as f:
    task = json.load(f)

print("loaded", task_path, {k: len(task[k]) for k in task})

loaded /kaggle/input/competitions/neurogolf-2026/task165.json {'train': 3, 'test': 1, 'arc-gen': 261}


In [7]:
def grid_to_onehot_30(grid):
    """Active canvas cells are one-hot, padded area is all-zero."""
    g = np.asarray(grid, dtype=np.int64)
    x = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = g.shape
    for r in range(h):
        for c in range(w):
            x[0, g[r, c], r, c] = 1.0
    return x

def onehot_to_grid(y):
    y = np.asarray(y)
    if y.ndim == 4:
        y = y[0]
    active = y.sum(axis=0) > 0.5
    pred = y.argmax(axis=0).astype(np.int64)
    pred[~active] = 0
    return pred

def expected_onehot_30(grid):
    return grid_to_onehot_30(grid)

def raw_padded_match(pred_onehot, expected_grid):
    exp = expected_onehot_30(expected_grid)
    return np.array_equal(pred_onehot, exp)

print("helpers ready")

helpers ready


In [8]:
class Task165TemplateCanvasMask(nn.Module):
    """Template-based symbolic model for task165.

    The object is the 4×7 hollow triangle:
        ...X...
        ..XXX..
        .XX.XX.
        X.....X

    The model:
    1. finds the object color by exact template convolution over non-background channels;
    2. treats the other non-background color as the marker/fill color;
    3. for each object-containing column, checks whether a marker exists below the object's
       bottommost pixel in that column;
    4. fills those selected columns downward, but only inside the active canvas mask.
    """
    def __init__(self, h=30, w=30):
        super().__init__()
        template = torch.tensor([
            [0,0,0,1,0,0,0],
            [0,0,1,1,1,0,0],
            [0,1,1,0,1,1,0],
            [1,0,0,0,0,0,1],
        ], dtype=torch.float32)
        self.register_buffer("template_weight", template.view(1,1,4,7).repeat(9,1,1,1))
        self.register_buffer("rows", torch.arange(h, dtype=torch.float32).view(1,1,h,1))

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).to(x.dtype)
        nonbg = x[:, 1:, :, :]

        score_map = F.conv2d(nonbg, self.template_weight, groups=9)
        scores = score_map.amax(dim=(2,3), keepdim=True)
        best = scores.amax(dim=1, keepdim=True)
        obj_sel = (scores >= best - 0.25).to(x.dtype)

        obj_mask = (nonbg * obj_sel).sum(dim=1, keepdim=True)
        bottom = (obj_mask * self.rows).amax(dim=2, keepdim=True)
        obj_col = (obj_mask.amax(dim=2, keepdim=True) > 0.5).to(x.dtype)

        below = (self.rows > bottom + 0.25).to(x.dtype)
        marker = nonbg * (1.0 - obj_sel)
        marker_below_col = ((marker * below).amax(dim=2, keepdim=True) > 0.5).to(x.dtype)

        selected_col = marker_below_col * obj_col
        fill = selected_col * below * active

        out_nonbg = torch.maximum(nonbg, fill)
        occupied = out_nonbg.amax(dim=1, keepdim=True)
        bg = active * (1.0 - occupied)
        return torch.cat([bg, out_nonbg], dim=1)

model = Task165TemplateCanvasMask(H, W).eval()
print(model)

Task165TemplateCanvasMask()


In [9]:
# Export static ONNX.
onnx_path = f"{TASK_ID}.onnx"
dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, onnx_path)

print("exported", onnx_path, "size", os.path.getsize(onnx_path))

/tmp/ipykernel_16/1161774462.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


exported task165.onnx size 6793


In [10]:
# ONNX graph checks.
onnx_model = onnx.load(onnx_path)
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden_present = sorted(FORBIDDEN_OPS.intersection(ops))
empty_optional_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]

def dims(value_info):
    return [
        d.dim_value if d.HasField("dim_value") else None
        for d in value_info.type.tensor_type.shape.dim
    ]

bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.output) + list(onnx_model.graph.value_info):
    if vi.type.HasField("tensor_type"):
        ds = dims(vi)
        if any(d is None or d == 0 for d in ds):
            bad_shapes.append((vi.name, ds))

print("ops:", dict(ops))
print("input:", [(i.name, dims(i)) for i in onnx_model.graph.input])
print("output:", [(o.name, dims(o)) for o in onnx_model.graph.output])
print("forbidden:", forbidden_present)
print("empty_optional_inputs:", len(empty_optional_inputs))
print("bad_shapes:", len(bad_shapes))

assert os.path.getsize(onnx_path) < 1_440_000
assert not forbidden_present
assert not empty_optional_inputs
assert not bad_shapes

ops: {'Constant': 12, 'ReduceSum': 2, 'Greater': 4, 'Cast': 5, 'Slice': 1, 'Conv': 1, 'ReduceMax': 6, 'Sub': 3, 'GreaterOrEqual': 1, 'Mul': 8, 'Add': 1, 'Max': 1, 'Concat': 1}
input: [('input', [1, 10, 30, 30])]
output: [('output', [1, 10, 30, 30])]
forbidden: []
empty_optional_inputs: 0
bad_shapes: 0


In [11]:
# ONNXRuntime raw-padded validation.
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

def predict_onehot(grid):
    x = grid_to_onehot_30(grid)
    return session.run(None, {"input": x})[0]

def validate_split(examples):
    flags = []
    for ex in examples:
        pred = predict_onehot(ex["input"])
        flags.append(raw_padded_match(pred, ex["output"]))
    return flags

validation = {}
for split in ["train", "test", "arc-gen"]:
    flags = validate_split(task[split])
    validation[split] = {
        "passed": int(sum(flags)),
        "total": len(flags),
        "failed_indices": [i for i, ok in enumerate(flags) if not ok][:20],
    }
    print(split, validation[split])

assert validation["train"]["passed"] == validation["train"]["total"]
assert validation["test"]["passed"] == validation["test"]["total"]
assert validation["arc-gen"]["passed"] == validation["arc-gen"]["total"]

train {'passed': 3, 'total': 3, 'failed_indices': []}
test {'passed': 1, 'total': 1, 'failed_indices': []}
arc-gen {'passed': 261, 'total': 261, 'failed_indices': []}


In [12]:
# Stronger grouped holdout validation: group by selected outlet-column signature.
template = np.array([
    [0,0,0,1,0,0,0],
    [0,0,1,1,1,0,0],
    [0,1,1,0,1,1,0],
    [1,0,0,0,0,0,1],
], dtype=np.int64)

def template_score_and_anchor(g, color):
    m = (g == color).astype(np.int64)
    th, tw = template.shape
    best, anchor = -1, (0, 0)
    for r in range(g.shape[0] - th + 1):
        for c in range(g.shape[1] - tw + 1):
            s = int((m[r:r+th, c:c+tw] * template).sum())
            if s > best:
                best, anchor = s, (r, c)
    return best, anchor

def structural_signature(ex):
    g = np.asarray(ex["input"], dtype=np.int64)
    colors = [int(c) for c in np.unique(g) if c != 0]
    scored = {c: template_score_and_anchor(g, c) for c in colors}
    obj_color = max(colors, key=lambda c: scored[c][0])
    marker_color = [c for c in colors if c != obj_color][0]
    anchor = scored[obj_color][1]
    obj = g == obj_color
    marker = g == marker_color
    rows = np.arange(g.shape[0])[:, None]
    bottom = (obj * rows).max(axis=0)
    obj_col = obj.max(axis=0)
    selected_cols = tuple(np.where(((marker & (rows > bottom[None, :])).max(axis=0)) & obj_col)[0].tolist())
    selected_rel = tuple(int(c - anchor[1]) for c in selected_cols)
    return (selected_rel, int(anchor[0] // 3), int(len(np.argwhere(marker)) // 5))

groups = {}
for i, ex in enumerate(task["arc-gen"]):
    groups.setdefault(structural_signature(ex), []).append(i)

keys = sorted(groups.keys(), key=str)
fit_idx, hold_idx = [], []
for j, k in enumerate(keys):
    (fit_idx if j % 2 == 0 else hold_idx).extend(groups[k])

arc_flags = validate_split(task["arc-gen"])
fit_ok = sum(arc_flags[i] for i in fit_idx)
hold_ok = sum(arc_flags[i] for i in hold_idx)

validation["arc_gen_grouped"] = {
    "num_groups": len(groups),
    "fit_passed": int(fit_ok),
    "fit_total": len(fit_idx),
    "holdout_passed": int(hold_ok),
    "holdout_total": len(hold_idx),
}

print(validation["arc_gen_grouped"])
assert fit_ok == len(fit_idx)
assert hold_ok == len(hold_idx)

{'num_groups': 217, 'fit_passed': 135, 'fit_total': 135, 'holdout_passed': 126, 'holdout_total': 126}


In [13]:
# Write validation summary and submission.zip.
validation_summary = {
    "task_id": TASK_ID,
    "modeling_approach": "4x7 hollow-triangle template convolution + active-canvas mask + marker-below-column fill",
    "input_shape": [1, CH, H, W],
    "output_shape": [1, CH, H, W],
    "onnx_size_bytes": os.path.getsize(onnx_path),
    "ops": dict(ops),
    "forbidden_ops_present": forbidden_present,
    "empty_optional_inputs": len(empty_optional_inputs),
    "bad_static_shapes": len(bad_shapes),
    "validation": validation,
}

summary_path = f"{TASK_ID}_30x30_validation_summary.json"
with open(summary_path, "w") as f:
    json.dump(validation_summary, f, indent=2)

submission_path = "submission.zip"
with zipfile.ZipFile(submission_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path, arcname=f"{TASK_ID}.onnx")

print("wrote", summary_path)
print("wrote", submission_path)
print("zip contents:", zipfile.ZipFile(submission_path).namelist())

wrote task165_30x30_validation_summary.json
wrote submission.zip
zip contents: ['task165.onnx']
